# 01.8 CPython vs PyPy and Other Implementations

Python is a SPECIFICATION, not a program. Several different programs
implement that specification, with different trade-offs.

This file detects which one you are running and explains the differences.


## PART 1: Which implementation are you running?


In [ ]:
# The platform module reports facts about the environment.
import platform

# sys gives lower-level interpreter details.
import sys

# python_implementation() names the implementation: CPython, PyPy, Jython...
implementation_name = platform.python_implementation()
print("Implementation:", implementation_name)

# The version of the Python LANGUAGE this implementation supports.
print("Language version:", platform.python_version())

# sys.implementation holds structured details about the same thing.
print("Implementation name (from sys):", sys.implementation.name)

# The cache tag stamped onto .pyc files, so different interpreters
# never read each other's cached bytecode.
print("Bytecode cache tag:", sys.implementation.cache_tag)

# Report what that means for the reader, in plain terms.
if implementation_name == "CPython":
    print("You are on CPython - the reference implementation, written in C.")
    print("This is what over 95% of the Python world runs, and what this")
    print("course targets. Good default; no reason to change.")
elif implementation_name == "PyPy":
    print("You are on PyPy - the JIT-compiled implementation.")
    print("Often much faster for long-running pure-Python work, but with")
    print("weaker C extension support. Unusual for learning.")
else:
    print("You are on", implementation_name, "- an alternative implementation.")
    print("Most of this course still applies, but some C-extension libraries")
    print("may behave differently or be unavailable.")

## PART 2: The GIL - present in CPython, and why it matters


In [ ]:
# The GIL is a lock inside CPython that allows only one thread to execute
# Python bytecode at a time. It makes memory management simpler and
# single-threaded code faster, at the cost of true CPU parallelism.

# Newer Python versions expose a way to check whether the GIL is active.
# We use hasattr() to test for the function before calling it, because
# it does not exist on older versions.
if hasattr(sys, "_is_gil_enabled"):
    # Python 3.13+ can be built without the GIL, so ask directly.
    gil_active = sys._is_gil_enabled()
    print("GIL currently enabled?", gil_active)
else:
    # On older versions the GIL is always present in CPython.
    print("This version has no GIL toggle - on CPython the GIL is always on.")

print("What the GIL means in practice:")
print("  - Threads do NOT give you CPU parallelism in CPython.")
print("  - Threads DO help when waiting on network or disk (I/O-bound work).")
print("  - For CPU-bound parallelism, use multiprocessing instead.")
print("Full treatment in Chapter 33 (threading) and Chapter 34 (multiprocessing).")

## PART 3: The implementations, compared


In [ ]:
# Each entry is (name, written in, best for, main limitation).
implementations = [
    (
        "CPython",
        "C",
        "everything - the default, best library support",
        "not the fastest; has the GIL",
    ),
    (
        "PyPy",
        "RPython (Python)",
        "long-running CPU-bound pure-Python code (often 4-10x faster)",
        "more memory, slower startup, weaker C extension support",
    ),
    (
        "Jython",
        "Java",
        "calling Java libraries from Python on the JVM",
        "lags behind on Python versions",
    ),
    (
        "IronPython",
        "C#",
        "calling .NET libraries from Python",
        "small community, lags on versions",
    ),
    (
        "MicroPython",
        "C",
        "microcontrollers and embedded hardware projects",
        "a subset of the language and library",
    ),
    (
        "GraalPy",
        "Java (GraalVM)",
        "polyglot applications mixing several languages",
        "newer, smaller ecosystem",
    ),
]

# Print each one as a readable block.
for name, written_in, best_for, limitation in implementations:
    print("")
    print("  " + name)
    print("     written in: ", written_in)
    print("     best for:   ", best_for)
    print("     limitation: ", limitation)

## PART 4: Names that get confused


In [ ]:
print("  Python   - the language specification (a document)")
print("  CPython  - the standard implementation, written in C")
print("  Cython   - a DIFFERENT thing: a Python superset that compiles to C")
print("  PyPy     - an implementation with a JIT compiler")
print("  PVM      - the Python Virtual Machine inside CPython, runs bytecode")
print("CPython and Cython being one letter apart is an unfortunate")
print("accident of naming. They are unrelated tools with different jobs.")

## PART 5: Writing code that works across implementations


In [ ]:
# Most Python code runs anywhere without changes. The exceptions are
# things that depend on CPython's internal behaviour.

# SAFE: standard language features, used the documented way.
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
sorted_numbers = sorted(numbers)
print("Sorting works identically everywhere:", sorted_numbers)

# RISKY: relying on CPython's reference-counting cleanup timing.
# CPython frees an object the instant its last reference disappears.
# PyPy uses a different garbage collector and may free it LATER.
print("Do NOT rely on this CPython-specific behaviour:")
print("  - objects being destroyed the moment the last reference goes")
print("  - files closing automatically when a variable goes out of scope")
print("  - the exact memory sizes reported by sys.getsizeof()")
print("Instead, close resources explicitly with `with` (Chapter 22).")
print("That works correctly on every implementation.")

# Here is how to write a check when you genuinely need one.
if platform.python_implementation() == "CPython":
    # Only run CPython-specific code inside a guard like this.
    print("")
    print("Running a CPython-only code path (guarded, so it stays portable).")

## PART 6: Which should you use?


In [ ]:
print("Use CPython from python.org. Full stop, until you have a measured")
print("reason to do otherwise.")
print("Consider PyPy only when ALL of these are true:")
print("  1. Your program is CPU-bound, not waiting on I/O.")
print("  2. It runs long enough for the JIT to warm up.")
print("  3. It is mostly pure Python, not C extensions.")
print("  4. You have PROFILED and confirmed CPython is the bottleneck.")
print("Point 4 matters most. Read Chapter 37 on profiling first - the")
print("bottleneck is almost never where people guess it is.")

## TAKEAWAYS


In [ ]:
print("TAKEAWAYS")
print("1. Python is a specification; several programs implement it.")
print("2. CPython is the reference implementation and the safe default.")
print("3. PyPy uses a JIT and can be far faster for CPU-bound pure Python.")
print("4. The GIL prevents CPU parallelism across threads in CPython.")
print("5. Cython is not CPython - it compiles Python-like code to C.")
print("6. Avoid depending on CPython internals so your code stays portable.")
print("7. Profile before switching implementations for speed.")

## TRY IT YOURSELF

1. Run this file and note your implementation. Almost certainly CPython.

2. Run `python -VV` in your terminal. What extra build information does
   the second V add?

3. Look at sys.implementation in the interpreter. What other fields does
   it hold besides name and cache_tag?

4. Find the __pycache__ folder in a project you have imported modules in.
   Does the filename contain the cache tag printed in Part 1?
